In [1]:
import os
import pandas as pd

# Inserisci il percorso della cartella se i file non sono nella stessa cartella del notebook
gtfs_dir = "."  # "." significa cartella corrente

print("Caricamento dei file GTFS...")
routes = pd.read_csv(os.path.join(gtfs_dir, 'routes.txt'))
trips = pd.read_csv(os.path.join(gtfs_dir, 'trips.txt'))
stop_times = pd.read_csv(os.path.join(gtfs_dir, 'stop_times.txt'))
stops = pd.read_csv(os.path.join(gtfs_dir, 'stops.txt'))

calendar_path = os.path.join(gtfs_dir, 'calendar.txt')
has_calendar = os.path.exists(calendar_path)
if has_calendar:
    calendar = pd.read_csv(calendar_path)

print("File caricati con successo!")

Caricamento dei file GTFS...
File caricati con successo!


In [3]:
# Ordinamento delle fermate
stop_times['stop_sequence'] = pd.to_numeric(stop_times['stop_sequence'])
stop_times = stop_times.sort_values(['trip_id', 'stop_sequence'])

# Prima e ultima fermata
first_stops = stop_times.groupby('trip_id').first().reset_index()
first_stops = first_stops[['trip_id', 'stop_id', 'departure_time']]
first_stops.rename(columns={'stop_id': 'start_stop_id', 'departure_time': 'departure_time'}, inplace=True)

last_stops = stop_times.groupby('trip_id').last().reset_index()
last_stops = last_stops[['trip_id', 'stop_id', 'arrival_time']]
last_stops.rename(columns={'stop_id': 'end_stop_id', 'arrival_time': 'arrival_time'}, inplace=True)

trip_endpoints = pd.merge(first_stops, last_stops, on='trip_id')

# Merge con le anagrafiche
trip_endpoints = pd.merge(trip_endpoints, stops[['stop_id', 'stop_name']], left_on='start_stop_id', right_on='stop_id', how='left')
trip_endpoints.rename(columns={'stop_name': 'fermata_partenza'}, inplace=True)
trip_endpoints.drop(columns=['stop_id', 'start_stop_id'], inplace=True)

trip_endpoints = pd.merge(trip_endpoints, stops[['stop_id', 'stop_name']], left_on='end_stop_id', right_on='stop_id', how='left')
trip_endpoints.rename(columns={'stop_name': 'fermata_arrivo'}, inplace=True)
trip_endpoints.drop(columns=['stop_id', 'end_stop_id'], inplace=True)

trip_full = pd.merge(trip_endpoints, trips, on='trip_id', how='left')

if 'route_short_name' in routes.columns:
    routes['linea'] = routes['route_short_name'].fillna(routes['route_long_name'])
else:
    routes['linea'] = routes['route_long_name']

trip_full = pd.merge(trip_full, routes[['route_id', 'linea']], on='route_id', how='left')

In [4]:
import pandas as pd

print("⏳ [1/3] Ordinamento delle fermate in corso...")
stop_times['stop_sequence'] = pd.to_numeric(stop_times['stop_sequence'])
stop_times = stop_times.sort_values(['trip_id', 'stop_sequence'])
print("   -> Ordinamento completato.")

print("⏳ [2/3] Estrazione di partenza e arrivo per ogni corsa...")
first_stops = stop_times.groupby('trip_id').first().reset_index()
first_stops = first_stops[['trip_id', 'stop_id', 'departure_time']]
first_stops.rename(columns={'stop_id': 'start_stop_id', 'departure_time': 'departure_time'}, inplace=True)

last_stops = stop_times.groupby('trip_id').last().reset_index()
last_stops = last_stops[['trip_id', 'stop_id', 'arrival_time']]
last_stops.rename(columns={'stop_id': 'end_stop_id', 'arrival_time': 'arrival_time'}, inplace=True)

trip_endpoints = pd.merge(first_stops, last_stops, on='trip_id')
print("   -> Capolinea di partenza e arrivo estratti.")

print("⏳ [3/3] Unione con i nomi delle fermate e delle linee...")
trip_endpoints = pd.merge(trip_endpoints, stops[['stop_id', 'stop_name']], left_on='start_stop_id', right_on='stop_id', how='left')
trip_endpoints.rename(columns={'stop_name': 'fermata_partenza'}, inplace=True)
trip_endpoints.drop(columns=['stop_id', 'start_stop_id'], inplace=True)

trip_endpoints = pd.merge(trip_endpoints, stops[['stop_id', 'stop_name']], left_on='end_stop_id', right_on='stop_id', how='left')
trip_endpoints.rename(columns={'stop_name': 'fermata_arrivo'}, inplace=True)
trip_endpoints.drop(columns=['stop_id', 'end_stop_id'], inplace=True)

trip_full = pd.merge(trip_endpoints, trips, on='trip_id', how='left')

if 'route_short_name' in routes.columns:
    routes['linea'] = routes['route_short_name'].fillna(routes['route_long_name'])
else:
    routes['linea'] = routes['route_long_name']
    
trip_full = pd.merge(trip_full, routes[['route_id', 'linea']], on='route_id', how='left')

print("✅ Secondo blocco completato con successo! Puoi passare al blocco successivo.")


⏳ [1/3] Ordinamento delle fermate in corso...
   -> Ordinamento completato.
⏳ [2/3] Estrazione di partenza e arrivo per ogni corsa...
   -> Capolinea di partenza e arrivo estratti.
⏳ [3/3] Unione con i nomi delle fermate e delle linee...
✅ Secondo blocco completato con successo! Puoi passare al blocco successivo.


In [5]:
if has_calendar:
    days = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
    days_it = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom']

    def get_active_days(row):
        active = [name for d, name in zip(days, days_it) if row.get(d) == 1]
        return ", ".join(active) if active else "Personalizzato"

    calendar['giorni_validita'] = calendar.apply(get_active_days, axis=1)
    trip_full = pd.merge(trip_full, calendar[['service_id', 'giorni_validita', 'start_date', 'end_date']], on='service_id', how='left')
else:
    trip_full['giorni_validita'] = "Non specificato"

output_columns = ['linea', 'trip_id', 'trip_headsign', 'fermata_partenza', 'departure_time', 'fermata_arrivo', 'arrival_time', 'giorni_validita']
existing_cols = [col for col in output_columns if col in trip_full.columns]
result_df = trip_full[existing_cols]

# Salvataggio in Excel
output_file = "riepilogo_corse.xlsx"
result_df.to_excel(output_file, index=False)
print(f"File Excel salvato con successo come: {output_file}")

# Mostra le prime 5 righe direttamente qui nel Notebook
display(result_df.head())

File Excel salvato con successo come: riepilogo_corse.xlsx


,linea,trip_id,trip_headsign,fermata_partenza,departure_time,fermata_arrivo,arrival_time,giorni_validita
0,436,ITRP-TR000f16b2-2a6c-4a28-bed0-f09c33904c87_Ex...,NaN,BUSSOLENO - PIAZZA STAZIONE - VIA TRAFORO,13:00:00,SUSA - PIAZZA IV NOVEMBRE,13:10:00,"Lun, Gio"
1,436,ITRP-TR000f16b2-2a6c-4a28-bed0-f09c33904c87_Ex...,NaN,BUSSOLENO - PIAZZA STAZIONE - VIA TRAFORO,13:00:00,SUSA - PIAZZA IV NOVEMBRE,13:10:00,"Lun, Gio"
2,436,ITRP-TR000f16b2-2a6c-4a28-bed0-f09c33904c87_Ex...,NaN,BUSSOLENO - PIAZZA STAZIONE - VIA TRAFORO,13:00:00,SUSA - PIAZZA IV NOVEMBRE,13:10:00,"Lun, Gio"
3,436,ITRP-TR000f16b2-2a6c-4a28-bed0-f09c33904c87_Ex...,NaN,BUSSOLENO - PIAZZA STAZIONE - VIA TRAFORO,13:00:00,SUSA - PIAZZA IV NOVEMBRE,13:10:00,"Lun, Gio"
4,3971,ITRP-TR00184108-5d21-4f8a-855b-3cbad3e32617_SF...,NaN,GERMAGNANO - STAZIONE F.T.C.,18:58:00,CERES - STAZIONE F.T.C.,19:19:00,Dom
